# Literature Recreation on Cleaned ILPD Dataset

This notebook recreates literature-inspired machine learning pipelines on the cleaned Indian Liver Patient Dataset (ILPD) under a unified benchmark framework.

- No noise is introduced.
- No new models are added.
- This is a benchmark notebook.
- Cross-validation, preprocessing, and holdout evaluation are standardized across all experiments.


In [1]:
import json
import os
import warnings
from pathlib import Path

os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib')
os.environ.setdefault('LOKY_MAX_CPU_COUNT', '1')

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from imblearn.ensemble import BalancedBaggingClassifier, EasyEnsembleClassifier, RUSBoostClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.ensemble import (
    AdaBoostClassifier,
    BaggingClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
    RandomForestClassifier,
    StackingClassifier,
)
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import ConstantKernel, RBF
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold, cross_validate, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
    XGBOOST_IMPORT_ERROR = None
except Exception as exc:
    XGBClassifier = None
    XGBOOST_AVAILABLE = False
    XGBOOST_IMPORT_ERROR = repr(exc)

warnings.filterwarnings('ignore')

RANDOM_STATE = 42
CV_SPLITS = 5
TEST_SIZE = 0.20
N_JOBS = 1
np.random.seed(RANDOM_STATE)
pd.options.display.float_format = '{:.4f}'.format
plt.style.use('default')

ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / 'pyproject.toml').exists() or (candidate / '.git').exists():
        ROOT = candidate
        break
else:
    raise FileNotFoundError('Could not locate repository root from current working directory')

DATASET_PATH = ROOT / 'data' / 'interim' / 'ILPD_cleaned.csv'
OUTPUT_DIR = ROOT / 'output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_PATH = OUTPUT_DIR / 'literature_recreation_cleaned_ilpd.csv'
BEST_MODEL_PATH = OUTPUT_DIR / 'best_literature_recreation_cleaned_ilpd.pkl'

SCORING = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'roc_auc': 'roc_auc',
}

CV_STRATEGY = StratifiedKFold(n_splits=CV_SPLITS, shuffle=True, random_state=RANDOM_STATE)

print(
    {
        'dataset_path': str(DATASET_PATH),
        'output_dir': str(OUTPUT_DIR),
        'cv_splits': CV_SPLITS,
        'test_size': TEST_SIZE,
        'random_state': RANDOM_STATE,
        'xgboost_available': XGBOOST_AVAILABLE,
    }
)
if not XGBOOST_AVAILABLE:
    print({'xgboost_status': 'optional dependency unavailable', 'details': XGBOOST_IMPORT_ERROR})


{'dataset_path': '/Users/samyabrataroy/Downloads/Spring_Internship_2026/data/interim/ILPD_cleaned.csv', 'output_dir': '/Users/samyabrataroy/Downloads/Spring_Internship_2026/output', 'cv_splits': 5, 'test_size': 0.2, 'random_state': 42, 'xgboost_available': True}


## Cleaned Dataset Loading


In [2]:
df = pd.read_csv(DATASET_PATH)

print(f'Dataset path: {DATASET_PATH}')
print(f'Shape: {df.shape}')
print('Columns:')
print(df.columns.tolist())
display(df.head())


Dataset path: /Users/samyabrataroy/Downloads/Spring_Internship_2026/data/interim/ILPD_cleaned.csv
Shape: (570, 11)
Columns:
['Age', 'Gender', 'Total_Bilirubin', 'Direct_Bilirubin', 'Alkaline_Phosphotase', 'Alamine_Aminotransferase', 'Aspartate_Aminotransferase', 'Total_Proteins', 'Albumin', 'Albumin_and_Globulin_Ratio', 'Result']


,Age,Gender,Total_Bilirubin,Direct_Bilirubin,Alkaline_Phosphotase,Alamine_Aminotransferase,Aspartate_Aminotransferase,Total_Proteins,Albumin,Albumin_and_Globulin_Ratio,Result
0,65,1,0.7000,0.1000,187,16,18,6.8000,3.3000,0.9000,1
1,62,0,10.9000,5.5000,699,64,100,7.5000,3.2000,0.7400,1
2,62,0,7.3000,4.1000,490,60,68,7.0000,3.3000,0.8900,1
3,58,0,1.0000,0.4000,182,14,20,6.8000,3.4000,1.0000,1
4,72,0,3.9000,2.0000,195,27,59,7.3000,2.4000,0.4000,1


## Target and Feature Setup


In [3]:
TARGET_COLUMN = 'Result'
FEATURE_COLUMNS = [column for column in df.columns if column != TARGET_COLUMN]

data = df.copy()
if 'Gender' in data.columns and data['Gender'].dtype == object:
    gender_map = {'male': 1, 'female': 0, 'm': 1, 'f': 0}
    data['Gender'] = data['Gender'].astype(str).str.strip().str.lower().map(gender_map)

for column in FEATURE_COLUMNS:
    if data[column].dtype == object:
        data[column], _ = pd.factorize(data[column].astype(str))

X = data[FEATURE_COLUMNS].apply(pd.to_numeric, errors='coerce').copy()
y_raw = pd.to_numeric(data[TARGET_COLUMN], errors='coerce')

valid_mask = y_raw.notna()
X = X.loc[valid_mask].reset_index(drop=True)
y_raw = y_raw.loc[valid_mask].astype(int).reset_index(drop=True)

class_labels = sorted(y_raw.unique().tolist())
if len(class_labels) != 2:
    raise ValueError(f'Expected binary target, found classes: {class_labels}')

target_mapping = {label: index for index, label in enumerate(class_labels)}
y = y_raw.map(target_mapping).astype(int)
NUMERIC_FEATURES = X.columns.tolist()

TARGET_MAPPING = pd.DataFrame(
    {
        'original_target': class_labels,
        'encoded_target': [target_mapping[label] for label in class_labels],
    }
)

display(TARGET_MAPPING)
display(pd.DataFrame({'target_encoded': y.value_counts().sort_index()}))
print(f'Feature matrix shape: {X.shape}')


,original_target,encoded_target
0,1,0
1,2,1


,target_encoded
Result,
0,406
1,164


Feature matrix shape: (570, 10)


## Shared Preprocessing Utilities


In [4]:
def make_preprocessor(scale_features: bool) -> ColumnTransformer:
    numeric_steps = [('imputer', SimpleImputer(strategy='median'))]
    if scale_features:
        numeric_steps.append(('scaler', StandardScaler()))

    return ColumnTransformer(
        transformers=[('numeric', Pipeline(steps=numeric_steps), NUMERIC_FEATURES)],
        remainder='drop',
        verbose_feature_names_out=False,
    )


def make_model_pipeline(estimator, scale_features: bool, use_smote: bool):
    steps = [('preprocessor', make_preprocessor(scale_features=scale_features))]
    if use_smote:
        steps.append(('smote', SMOTE(random_state=RANDOM_STATE)))
    steps.append(('model', estimator))
    return ImbPipeline(steps=steps)


PREPROCESSING_REGISTRY = pd.DataFrame(
    [
        {'pipeline_type': 'No scaling', 'scaling': False, 'smote': False},
        {'pipeline_type': 'With scaling', 'scaling': True, 'smote': False},
        {'pipeline_type': 'With SMOTE', 'scaling': False, 'smote': True},
        {'pipeline_type': 'With scaling + SMOTE', 'scaling': True, 'smote': True},
    ]
)

display(PREPROCESSING_REGISTRY)


,pipeline_type,scaling,smote
0,No scaling,False,False
1,With scaling,True,False
2,With SMOTE,False,True
3,With scaling + SMOTE,True,True


## Evaluation Utilities


In [5]:
RESULTS_REGISTRY = []
FITTED_MODEL_REGISTRY = {}
SEARCH_REGISTRY = {}


def get_positive_scores(fitted_estimator, X_eval):
    if hasattr(fitted_estimator, 'predict_proba'):
        return fitted_estimator.predict_proba(X_eval)[:, 1]
    if hasattr(fitted_estimator, 'decision_function'):
        scores = fitted_estimator.decision_function(X_eval)
        if np.ndim(scores) > 1:
            return scores[:, 1]
        return scores
    return None


def compute_holdout_metrics(fitted_estimator, X_eval, y_eval):
    y_pred = fitted_estimator.predict(X_eval)
    y_score = get_positive_scores(fitted_estimator, X_eval)

    metrics = {
        'Holdout Accuracy': accuracy_score(y_eval, y_pred),
        'Holdout Precision': precision_score(y_eval, y_pred, zero_division=0),
        'Holdout Recall': recall_score(y_eval, y_pred, zero_division=0),
        'Holdout F1': f1_score(y_eval, y_pred, zero_division=0),
        'Holdout ROC-AUC': np.nan,
    }
    if y_score is not None:
        metrics['Holdout ROC-AUC'] = roc_auc_score(y_eval, y_score)
    return metrics


def summarize_cv_scores(cv_output: dict) -> dict:
    return {
        'CV Accuracy': float(np.nanmean(cv_output['test_accuracy'])),
        'CV Precision': float(np.nanmean(cv_output['test_precision'])),
        'CV Recall': float(np.nanmean(cv_output['test_recall'])),
        'CV F1': float(np.nanmean(cv_output['test_f1'])),
        'CV ROC-AUC': float(np.nanmean(cv_output['test_roc_auc'])),
    }


def empty_result_row(spec: dict, notes: str) -> dict:
    return {
        'Registry Key': spec['registry_key'],
        'Section': spec['section'],
        'Model': spec['model'],
        'Variant': spec['variant'],
        'Scaling Used': spec['scaling_used'],
        'SMOTE Used': spec['smote_used'],
        'Tuned': spec['tuned'],
        'CV Accuracy': np.nan,
        'CV Precision': np.nan,
        'CV Recall': np.nan,
        'CV F1': np.nan,
        'CV ROC-AUC': np.nan,
        'Holdout Accuracy': np.nan,
        'Holdout Precision': np.nan,
        'Holdout Recall': np.nan,
        'Holdout F1': np.nan,
        'Holdout ROC-AUC': np.nan,
        'ROC-AUC': np.nan,
        'Notes': notes,
    }


def evaluate_model_spec(spec: dict, X_train, y_train, X_test, y_test) -> dict:
    if spec.get('skip', False):
        row = empty_result_row(spec, spec['notes'])
        RESULTS_REGISTRY.append(row)
        return row

    try:
        if spec['tuned'] == 'Yes':
            search = spec['search_builder']()
            search.fit(X_train, y_train)
            best_estimator = search.best_estimator_
            SEARCH_REGISTRY[spec['registry_key']] = search
            cv_output = cross_validate(
                best_estimator,
                X_train,
                y_train,
                cv=CV_STRATEGY,
                scoring=SCORING,
                n_jobs=N_JOBS,
                error_score='raise',
            )
            fitted_estimator = search.best_estimator_
            best_params_text = json.dumps(search.best_params_, sort_keys=True)
            notes = spec['notes'] + f' | best_params={best_params_text}'
        else:
            estimator = spec['builder']()
            cv_output = cross_validate(
                estimator,
                X_train,
                y_train,
                cv=CV_STRATEGY,
                scoring=SCORING,
                n_jobs=N_JOBS,
                error_score='raise',
            )
            fitted_estimator = clone(estimator)
            fitted_estimator.fit(X_train, y_train)
            notes = spec['notes']

        holdout_metrics = compute_holdout_metrics(fitted_estimator, X_test, y_test)
        cv_metrics = summarize_cv_scores(cv_output)

        row = {
            'Registry Key': spec['registry_key'],
            'Section': spec['section'],
            'Model': spec['model'],
            'Variant': spec['variant'],
            'Scaling Used': spec['scaling_used'],
            'SMOTE Used': spec['smote_used'],
            'Tuned': spec['tuned'],
            **cv_metrics,
            **holdout_metrics,
            'ROC-AUC': holdout_metrics['Holdout ROC-AUC'],
            'Notes': notes,
        }
        FITTED_MODEL_REGISTRY[spec['registry_key']] = fitted_estimator
    except Exception as exc:
        row = empty_result_row(spec, spec['notes'] + f' | failed={type(exc).__name__}: {exc}')

    RESULTS_REGISTRY.append(row)
    return row


def run_block(block_specs, X_train, y_train, X_test, y_test):
    block_rows = [evaluate_model_spec(spec, X_train, y_train, X_test, y_test) for spec in block_specs]
    block_df = pd.DataFrame(block_rows)
    display(
        block_df[
            [
                'Section',
                'Model',
                'Variant',
                'Scaling Used',
                'SMOTE Used',
                'Tuned',
                'CV F1',
                'Holdout F1',
                'ROC-AUC',
                'Notes',
            ]
        ]
    )
    return block_df


## Train/Test Split


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

SPLIT_SUMMARY = pd.DataFrame(
    [
        {
            'split': 'train',
            'rows': int(len(X_train)),
            'class_0': int((y_train == 0).sum()),
            'class_1': int((y_train == 1).sum()),
        },
        {
            'split': 'test',
            'rows': int(len(X_test)),
            'class_0': int((y_test == 0).sum()),
            'class_1': int((y_test == 1).sum()),
        },
    ]
)

display(SPLIT_SUMMARY)


,split,rows,class_0,class_1
0,train,456,325,131
1,test,114,81,33


## Block 1: Hanif & Khan (Basic ML + SMOTE)


In [7]:
BLOCK_1_SPECS = [
    {
        'registry_key': 'block1_decision_tree_smote',
        'section': 'Block 1: Hanif & Khan',
        'model': 'Decision Tree',
        'variant': 'baseline',
        'scaling_used': 'None',
        'smote_used': 'Yes',
        'tuned': 'No',
        'notes': 'DecisionTreeClassifier + SMOTE recreation',
        'builder': lambda: make_model_pipeline(
            DecisionTreeClassifier(max_depth=5, min_samples_leaf=4, random_state=RANDOM_STATE),
            scale_features=False,
            use_smote=True,
        ),
    },
    {
        'registry_key': 'block1_random_forest_smote',
        'section': 'Block 1: Hanif & Khan',
        'model': 'Random Forest',
        'variant': 'baseline',
        'scaling_used': 'None',
        'smote_used': 'Yes',
        'tuned': 'No',
        'notes': 'RandomForestClassifier + SMOTE recreation',
        'builder': lambda: make_model_pipeline(
            RandomForestClassifier(
                n_estimators=200,
                min_samples_leaf=2,
                class_weight='balanced_subsample',
                random_state=RANDOM_STATE,
                n_jobs=N_JOBS,
            ),
            scale_features=False,
            use_smote=True,
        ),
    },
    {
        'registry_key': 'block1_svm_rbf_scaling_smote',
        'section': 'Block 1: Hanif & Khan',
        'model': 'SVM (RBF)',
        'variant': 'baseline',
        'scaling_used': 'StandardScaler',
        'smote_used': 'Yes',
        'tuned': 'No',
        'notes': 'SVC with RBF kernel + scaling + SMOTE recreation',
        'builder': lambda: make_model_pipeline(
            SVC(C=1.0, kernel='rbf', gamma='scale', probability=True, random_state=RANDOM_STATE),
            scale_features=True,
            use_smote=True,
        ),
    },
]

BLOCK_1_RESULTS = run_block(BLOCK_1_SPECS, X_train, y_train, X_test, y_test)


,Section,Model,Variant,Scaling Used,SMOTE Used,Tuned,CV F1,Holdout F1,ROC-AUC,Notes
0,Block 1: Hanif & Khan,Decision Tree,baseline,None,Yes,No,0.5314,0.4390,0.6178,DecisionTreeClassifier + SMOTE recreation
1,Block 1: Hanif & Khan,Random Forest,baseline,None,Yes,No,0.4892,0.4865,0.7531,RandomForestClassifier + SMOTE recreation
2,Block 1: Hanif & Khan,SVM (RBF),baseline,StandardScaler,Yes,No,0.5321,0.5591,0.7497,SVC with RBF kernel + scaling + SMOTE recreation


## Block 2: Makkena & Natarajan (Extended + Tuned Models)


In [8]:
BLOCK_2_SPECS = [
    {
        'registry_key': 'block2_logistic_regression_baseline',
        'section': 'Block 2: Makkena & Natarajan',
        'model': 'Logistic Regression',
        'variant': 'baseline',
        'scaling_used': 'StandardScaler',
        'smote_used': 'Yes',
        'tuned': 'No',
        'notes': 'Baseline LogisticRegression + scaling + SMOTE',
        'builder': lambda: make_model_pipeline(
            LogisticRegression(C=1.0, solver='liblinear', max_iter=5000, random_state=RANDOM_STATE),
            scale_features=True,
            use_smote=True,
        ),
    },
    {
        'registry_key': 'block2_naive_bayes_baseline',
        'section': 'Block 2: Makkena & Natarajan',
        'model': 'Naive Bayes',
        'variant': 'baseline',
        'scaling_used': 'None',
        'smote_used': 'Yes',
        'tuned': 'No',
        'notes': 'Baseline GaussianNB + SMOTE',
        'builder': lambda: make_model_pipeline(
            GaussianNB(),
            scale_features=False,
            use_smote=True,
        ),
    },
    {
        'registry_key': 'block2_random_forest_baseline',
        'section': 'Block 2: Makkena & Natarajan',
        'model': 'Random Forest',
        'variant': 'baseline',
        'scaling_used': 'None',
        'smote_used': 'Yes',
        'tuned': 'No',
        'notes': 'Baseline RandomForestClassifier + SMOTE',
        'builder': lambda: make_model_pipeline(
            RandomForestClassifier(
                n_estimators=200,
                min_samples_leaf=2,
                class_weight='balanced_subsample',
                random_state=RANDOM_STATE,
                n_jobs=N_JOBS,
            ),
            scale_features=False,
            use_smote=True,
        ),
    },
    {
        'registry_key': 'block2_bagging_baseline',
        'section': 'Block 2: Makkena & Natarajan',
        'model': 'Bagging Classifier',
        'variant': 'baseline',
        'scaling_used': 'None',
        'smote_used': 'Yes',
        'tuned': 'No',
        'notes': 'Baseline BaggingClassifier + SMOTE',
        'builder': lambda: make_model_pipeline(
            BaggingClassifier(
                estimator=DecisionTreeClassifier(min_samples_leaf=2, random_state=RANDOM_STATE),
                n_estimators=50,
                random_state=RANDOM_STATE,
                n_jobs=N_JOBS,
            ),
            scale_features=False,
            use_smote=True,
        ),
    },
    {
        'registry_key': 'block2_gradient_boosting_baseline',
        'section': 'Block 2: Makkena & Natarajan',
        'model': 'Gradient Boosting',
        'variant': 'baseline',
        'scaling_used': 'None',
        'smote_used': 'Yes',
        'tuned': 'No',
        'notes': 'Baseline GradientBoostingClassifier + SMOTE',
        'builder': lambda: make_model_pipeline(
            GradientBoostingClassifier(random_state=RANDOM_STATE),
            scale_features=False,
            use_smote=True,
        ),
    },
    {
        'registry_key': 'block2_adaboost_baseline',
        'section': 'Block 2: Makkena & Natarajan',
        'model': 'AdaBoost',
        'variant': 'baseline',
        'scaling_used': 'None',
        'smote_used': 'Yes',
        'tuned': 'No',
        'notes': 'Baseline AdaBoostClassifier + SMOTE',
        'builder': lambda: make_model_pipeline(
            AdaBoostClassifier(n_estimators=100, learning_rate=0.5, random_state=RANDOM_STATE),
            scale_features=False,
            use_smote=True,
        ),
    },
    {
        'registry_key': 'block2_xgboost_baseline',
        'section': 'Block 2: Makkena & Natarajan',
        'model': 'XGBoost',
        'variant': 'baseline',
        'scaling_used': 'None',
        'smote_used': 'Yes',
        'tuned': 'No',
        'notes': 'Baseline XGBClassifier + SMOTE' if XGBOOST_AVAILABLE else 'Skipped: xgboost not installed',
        'skip': not XGBOOST_AVAILABLE,
        'builder': (lambda: make_model_pipeline(
            XGBClassifier(
                n_estimators=200,
                max_depth=4,
                learning_rate=0.05,
                subsample=0.9,
                colsample_bytree=0.9,
                eval_metric='logloss',
                random_state=RANDOM_STATE,
                n_jobs=N_JOBS,
            ),
            scale_features=False,
            use_smote=True,
        )) if XGBOOST_AVAILABLE else None,
    },
    {
        'registry_key': 'block2_logistic_regression_tuned',
        'section': 'Block 2: Makkena & Natarajan',
        'model': 'Logistic Regression',
        'variant': 'tuned',
        'scaling_used': 'StandardScaler',
        'smote_used': 'Yes',
        'tuned': 'Yes',
        'notes': 'GridSearchCV over LogisticRegression C and penalty',
        'search_builder': lambda: GridSearchCV(
            estimator=make_model_pipeline(
                LogisticRegression(solver='liblinear', max_iter=5000, random_state=RANDOM_STATE),
                scale_features=True,
                use_smote=True,
            ),
            param_grid={
                'model__C': [0.1, 0.5, 1.0, 2.0, 5.0],
                'model__penalty': ['l1', 'l2'],
            },
            scoring='f1',
            cv=CV_STRATEGY,
            n_jobs=N_JOBS,
            refit=True,
        ),
    },
    {
        'registry_key': 'block2_random_forest_tuned',
        'section': 'Block 2: Makkena & Natarajan',
        'model': 'Random Forest',
        'variant': 'tuned',
        'scaling_used': 'None',
        'smote_used': 'Yes',
        'tuned': 'Yes',
        'notes': 'RandomizedSearchCV over RandomForestClassifier tree settings',
        'search_builder': lambda: RandomizedSearchCV(
            estimator=make_model_pipeline(
                RandomForestClassifier(
                    class_weight='balanced_subsample',
                    random_state=RANDOM_STATE,
                    n_jobs=N_JOBS,
                ),
                scale_features=False,
                use_smote=True,
            ),
            param_distributions={
                'model__n_estimators': [100, 200, 300],
                'model__max_depth': [None, 5, 10, 15],
                'model__min_samples_leaf': [1, 2, 4],
                'model__max_features': ['sqrt', 'log2', None],
            },
            n_iter=10,
            scoring='f1',
            cv=CV_STRATEGY,
            random_state=RANDOM_STATE,
            n_jobs=N_JOBS,
            refit=True,
        ),
    },
    {
        'registry_key': 'block2_xgboost_tuned',
        'section': 'Block 2: Makkena & Natarajan',
        'model': 'XGBoost',
        'variant': 'tuned',
        'scaling_used': 'None',
        'smote_used': 'Yes',
        'tuned': 'Yes',
        'notes': 'RandomizedSearchCV over XGBClassifier settings' if XGBOOST_AVAILABLE else 'Skipped: xgboost not installed',
        'skip': not XGBOOST_AVAILABLE,
        'search_builder': (lambda: RandomizedSearchCV(
            estimator=make_model_pipeline(
                XGBClassifier(
                    eval_metric='logloss',
                    random_state=RANDOM_STATE,
                    n_jobs=N_JOBS,
                ),
                scale_features=False,
                use_smote=True,
            ),
            param_distributions={
                'model__n_estimators': [100, 200, 300],
                'model__max_depth': [3, 4, 5],
                'model__learning_rate': [0.03, 0.05, 0.10],
                'model__subsample': [0.8, 0.9, 1.0],
                'model__colsample_bytree': [0.8, 0.9, 1.0],
            },
            n_iter=10,
            scoring='f1',
            cv=CV_STRATEGY,
            random_state=RANDOM_STATE,
            n_jobs=N_JOBS,
            refit=True,
        )) if XGBOOST_AVAILABLE else None,
    },
]

BLOCK_2_RESULTS = run_block(BLOCK_2_SPECS, X_train, y_train, X_test, y_test)


,Section,Model,Variant,Scaling Used,SMOTE Used,Tuned,CV F1,Holdout F1,ROC-AUC,Notes
0,Block 2: Makkena & Natarajan,Logistic Regression,baseline,StandardScaler,Yes,No,0.5458,0.5918,0.7826,Baseline LogisticRegression + scaling + SMOTE
1,Block 2: Makkena & Natarajan,Naive Bayes,baseline,None,Yes,No,0.5495,0.5536,0.7175,Baseline GaussianNB + SMOTE
2,Block 2: Makkena & Natarajan,Random Forest,baseline,None,Yes,No,0.4892,0.4865,0.7531,Baseline RandomForestClassifier + SMOTE
3,Block 2: Makkena & Natarajan,Bagging Classifier,baseline,None,Yes,No,0.4870,0.4324,0.7106,Baseline BaggingClassifier + SMOTE
4,Block 2: Makkena & Natarajan,Gradient Boosting,baseline,None,Yes,No,0.4396,0.5195,0.7400,Baseline GradientBoostingClassifier + SMOTE
5,Block 2: Makkena & Natarajan,AdaBoost,baseline,None,Yes,No,0.5340,0.5859,0.7344,Baseline AdaBoostClassifier + SMOTE
6,Block 2: Makkena & Natarajan,XGBoost,baseline,None,Yes,No,0.4486,0.4706,0.7209,Baseline XGBClassifier + SMOTE
7,Block 2: Makkena & Natarajan,Logistic Regression,tuned,StandardScaler,Yes,Yes,0.5645,0.5769,0.7856,GridSearchCV over LogisticRegression C and pen...
8,Block 2: Makkena & Natarajan,Random Forest,tuned,None,Yes,Yes,0.5685,0.5556,0.7314,RandomizedSearchCV over RandomForestClassifier...
9,Block 2: Makkena & Natarajan,XGBoost,tuned,None,Yes,Yes,0.5305,0.5476,0.7262,RandomizedSearchCV over XGBClassifier settings...


## Block 3: Ma & Li (Advanced + Ensemble Models)


In [9]:
BLOCK_3_SPECS = [
    {
        'registry_key': 'block3_qda_scaling',
        'section': 'Block 3: Ma & Li',
        'model': 'QDA',
        'variant': 'baseline',
        'scaling_used': 'StandardScaler',
        'smote_used': 'No',
        'tuned': 'No',
        'notes': 'QuadraticDiscriminantAnalysis + scaling',
        'builder': lambda: make_model_pipeline(
            QuadraticDiscriminantAnalysis(reg_param=0.1),
            scale_features=True,
            use_smote=False,
        ),
    },
    {
        'registry_key': 'block3_gradient_boosting',
        'section': 'Block 3: Ma & Li',
        'model': 'Gradient Boosting',
        'variant': 'baseline',
        'scaling_used': 'None',
        'smote_used': 'No',
        'tuned': 'No',
        'notes': 'GradientBoostingClassifier without SMOTE',
        'builder': lambda: make_model_pipeline(
            GradientBoostingClassifier(random_state=RANDOM_STATE),
            scale_features=False,
            use_smote=False,
        ),
    },
    {
        'registry_key': 'block3_hist_gradient_boosting',
        'section': 'Block 3: Ma & Li',
        'model': 'HistGradientBoosting',
        'variant': 'baseline',
        'scaling_used': 'None',
        'smote_used': 'No',
        'tuned': 'No',
        'notes': 'HistGradientBoostingClassifier without SMOTE',
        'builder': lambda: make_model_pipeline(
            HistGradientBoostingClassifier(max_depth=5, learning_rate=0.05, max_iter=150, random_state=RANDOM_STATE),
            scale_features=False,
            use_smote=False,
        ),
    },
    {
        'registry_key': 'block3_balanced_bagging',
        'section': 'Block 3: Ma & Li',
        'model': 'Balanced Bagging',
        'variant': 'baseline',
        'scaling_used': 'None',
        'smote_used': 'No',
        'tuned': 'No',
        'notes': 'BalancedBaggingClassifier imbalance-aware ensemble',
        'builder': lambda: make_model_pipeline(
            BalancedBaggingClassifier(
                estimator=DecisionTreeClassifier(max_depth=6, min_samples_leaf=4, random_state=RANDOM_STATE),
                n_estimators=25,
                random_state=RANDOM_STATE,
                n_jobs=N_JOBS,
            ),
            scale_features=False,
            use_smote=False,
        ),
    },
    {
        'registry_key': 'block3_easy_ensemble',
        'section': 'Block 3: Ma & Li',
        'model': 'Easy Ensemble',
        'variant': 'baseline',
        'scaling_used': 'None',
        'smote_used': 'No',
        'tuned': 'No',
        'notes': 'EasyEnsembleClassifier imbalance-aware ensemble',
        'builder': lambda: make_model_pipeline(
            EasyEnsembleClassifier(n_estimators=15, random_state=RANDOM_STATE, n_jobs=N_JOBS),
            scale_features=False,
            use_smote=False,
        ),
    },
    {
        'registry_key': 'block3_rusboost',
        'section': 'Block 3: Ma & Li',
        'model': 'RUSBoost',
        'variant': 'baseline',
        'scaling_used': 'None',
        'smote_used': 'No',
        'tuned': 'No',
        'notes': 'RUSBoostClassifier imbalance-aware boosting',
        'builder': lambda: make_model_pipeline(
            RUSBoostClassifier(
                estimator=DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE),
                n_estimators=100,
                learning_rate=0.5,
                random_state=RANDOM_STATE,
            ),
            scale_features=False,
            use_smote=False,
        ),
    },
    {
        'registry_key': 'block3_gaussian_process_scaling',
        'section': 'Block 3: Ma & Li',
        'model': 'Gaussian Process Classifier',
        'variant': 'baseline',
        'scaling_used': 'StandardScaler',
        'smote_used': 'No',
        'tuned': 'No',
        'notes': 'GaussianProcessClassifier + scaling',
        'builder': lambda: make_model_pipeline(
            GaussianProcessClassifier(
                kernel=ConstantKernel(1.0) * RBF(length_scale=1.0),
                random_state=RANDOM_STATE,
            ),
            scale_features=True,
            use_smote=False,
        ),
    },
    {
        'registry_key': 'block3_stacking_classifier',
        'section': 'Block 3: Ma & Li',
        'model': 'Stacking Classifier',
        'variant': 'baseline',
        'scaling_used': 'StandardScaler',
        'smote_used': 'No',
        'tuned': 'No',
        'notes': 'Stacking of Logistic Regression, Random Forest, and Gradient Boosting with Logistic Regression final estimator',
        'builder': lambda: make_model_pipeline(
            StackingClassifier(
                estimators=[
                    ('lr', LogisticRegression(C=1.0, solver='liblinear', max_iter=5000, random_state=RANDOM_STATE)),
                    ('rf', RandomForestClassifier(n_estimators=150, random_state=RANDOM_STATE, n_jobs=N_JOBS)),
                    ('gb', GradientBoostingClassifier(random_state=RANDOM_STATE)),
                ],
                final_estimator=LogisticRegression(C=1.0, solver='liblinear', max_iter=5000, random_state=RANDOM_STATE),
                cv=5,
                n_jobs=N_JOBS,
                stack_method='predict_proba',
            ),
            scale_features=True,
            use_smote=False,
        ),
    },
]

BLOCK_3_RESULTS = run_block(BLOCK_3_SPECS, X_train, y_train, X_test, y_test)


,Section,Model,Variant,Scaling Used,SMOTE Used,Tuned,CV F1,Holdout F1,ROC-AUC,Notes
0,Block 3: Ma & Li,QDA,baseline,StandardScaler,No,No,0.5405,0.5455,0.7546,QuadraticDiscriminantAnalysis + scaling
1,Block 3: Ma & Li,Gradient Boosting,baseline,None,No,No,0.3658,0.4000,0.7052,GradientBoostingClassifier without SMOTE
2,Block 3: Ma & Li,HistGradientBoosting,baseline,None,No,No,0.3977,0.3939,0.6891,HistGradientBoostingClassifier without SMOTE
3,Block 3: Ma & Li,Balanced Bagging,baseline,None,No,No,0.5571,0.5806,0.7351,BalancedBaggingClassifier imbalance-aware ense...
4,Block 3: Ma & Li,Easy Ensemble,baseline,None,No,No,0.5604,0.5859,0.7475,EasyEnsembleClassifier imbalance-aware ensemble
5,Block 3: Ma & Li,RUSBoost,baseline,None,No,No,0.4893,0.5714,0.7306,RUSBoostClassifier imbalance-aware boosting
6,Block 3: Ma & Li,Gaussian Process Classifier,baseline,StandardScaler,No,No,0.0000,0.0000,0.7950,GaussianProcessClassifier + scaling
7,Block 3: Ma & Li,Stacking Classifier,baseline,StandardScaler,No,No,0.0492,0.2051,0.7845,"Stacking of Logistic Regression, Random Forest..."


## Final Output


In [10]:
RESULTS_DETAILED = pd.DataFrame(RESULTS_REGISTRY)
RESULTS_DETAILED = RESULTS_DETAILED.sort_values(
    by=['Holdout F1', 'CV F1'],
    ascending=[False, False],
    na_position='last',
).reset_index(drop=True)

FINAL_RESULTS_TABLE = RESULTS_DETAILED[
    [
        'Section',
        'Model',
        'Variant',
        'Scaling Used',
        'SMOTE Used',
        'Tuned',
        'CV Accuracy',
        'CV F1',
        'Holdout Accuracy',
        'Holdout F1',
        'ROC-AUC',
        'Notes',
    ]
].copy()

BEST_MODEL_OVERALL = RESULTS_DETAILED.head(1)[
    ['Section', 'Model', 'Variant', 'CV F1', 'Holdout F1', 'ROC-AUC', 'Notes']
].reset_index(drop=True)

BEST_MODEL_BY_BLOCK = (
    RESULTS_DETAILED.sort_values(by=['Holdout F1', 'CV F1'], ascending=[False, False], na_position='last')
    .drop_duplicates(subset='Section')
    [['Section', 'Model', 'Variant', 'CV F1', 'Holdout F1', 'ROC-AUC', 'Notes']]
    .reset_index(drop=True)
)

display(RESULTS_DETAILED)
display(FINAL_RESULTS_TABLE)
display(BEST_MODEL_OVERALL)
display(BEST_MODEL_BY_BLOCK)

FINAL_RESULTS_TABLE.to_csv(RESULTS_PATH, index=False)

if not RESULTS_DETAILED.empty:
    best_registry_key = RESULTS_DETAILED.loc[0, 'Registry Key']
    if best_registry_key in FITTED_MODEL_REGISTRY:
        joblib.dump(FITTED_MODEL_REGISTRY[best_registry_key], BEST_MODEL_PATH)

print(
    {
        'results_csv': str(RESULTS_PATH),
        'best_model_pickle': str(BEST_MODEL_PATH),
        'best_model': None if BEST_MODEL_OVERALL.empty else BEST_MODEL_OVERALL.loc[0, 'Model'],
        'best_model_variant': None if BEST_MODEL_OVERALL.empty else BEST_MODEL_OVERALL.loc[0, 'Variant'],
    }
)


,Registry Key,Section,Model,Variant,Scaling Used,SMOTE Used,Tuned,CV Accuracy,CV Precision,CV Recall,CV F1,CV ROC-AUC,Holdout Accuracy,Holdout Precision,Holdout Recall,Holdout F1,Holdout ROC-AUC,ROC-AUC,Notes
0,block2_logistic_regression_baseline,Block 2: Makkena & Natarajan,Logistic Regression,baseline,StandardScaler,Yes,No,0.6273,0.4192,0.7860,0.5458,0.7448,0.6491,0.4462,0.8788,0.5918,0.7826,0.7826,Baseline LogisticRegression + scaling + SMOTE
1,block3_easy_ensemble,Block 3: Ma & Li,Easy Ensemble,baseline,None,No,No,0.6514,0.4370,0.7869,0.5604,0.7324,0.6404,0.4394,0.8788,0.5859,0.7475,0.7475,EasyEnsembleClassifier imbalance-aware ensemble
2,block2_adaboost_baseline,Block 2: Makkena & Natarajan,AdaBoost,baseline,None,Yes,No,0.6426,0.4255,0.7248,0.5340,0.7277,0.6404,0.4394,0.8788,0.5859,0.7344,0.7344,Baseline AdaBoostClassifier + SMOTE
3,block3_balanced_bagging,Block 3: Ma & Li,Balanced Bagging,baseline,None,No,No,0.6645,0.4497,0.7333,0.5571,0.7244,0.6579,0.4500,0.8182,0.5806,0.7351,0.7351,BalancedBaggingClassifier imbalance-aware ense...
4,block2_logistic_regression_tuned,Block 2: Makkena & Natarajan,Logistic Regression,tuned,StandardScaler,Yes,Yes,0.6118,0.4154,0.8849,0.5645,0.7392,0.6140,0.4225,0.9091,0.5769,0.7856,0.7856,GridSearchCV over LogisticRegression C and pen...
5,block3_rusboost,Block 3: Ma & Li,RUSBoost,baseline,None,No,No,0.6294,0.4020,0.6325,0.4893,0.6818,0.6842,0.4706,0.7273,0.5714,0.7306,0.7306,RUSBoostClassifier imbalance-aware boosting
6,block1_svm_rbf_scaling_smote,Block 1: Hanif & Khan,SVM (RBF),baseline,StandardScaler,Yes,No,0.6184,0.4095,0.7638,0.5321,0.7188,0.6404,0.4333,0.7879,0.5591,0.7497,0.7497,SVC with RBF kernel + scaling + SMOTE recreation
7,block2_random_forest_tuned,Block 2: Makkena & Natarajan,Random Forest,tuned,None,Yes,Yes,0.6756,0.4588,0.7484,0.5685,0.7246,0.6491,0.4386,0.7576,0.5556,0.7314,0.7314,RandomizedSearchCV over RandomForestClassifier...
8,block2_naive_bayes_baseline,Block 2: Makkena & Natarajan,Naive Bayes,baseline,None,Yes,No,0.5658,0.3914,0.9239,0.5495,0.7344,0.5614,0.3924,0.9394,0.5536,0.7175,0.7175,Baseline GaussianNB + SMOTE
9,block2_xgboost_tuned,Block 2: Makkena & Natarajan,XGBoost,tuned,None,Yes,Yes,0.6842,0.4618,0.6256,0.5305,0.7209,0.6667,0.4510,0.6970,0.5476,0.7262,0.7262,RandomizedSearchCV over XGBClassifier settings...


,Section,Model,Variant,Scaling Used,SMOTE Used,Tuned,CV Accuracy,CV F1,Holdout Accuracy,Holdout F1,ROC-AUC,Notes
0,Block 2: Makkena & Natarajan,Logistic Regression,baseline,StandardScaler,Yes,No,0.6273,0.5458,0.6491,0.5918,0.7826,Baseline LogisticRegression + scaling + SMOTE
1,Block 3: Ma & Li,Easy Ensemble,baseline,None,No,No,0.6514,0.5604,0.6404,0.5859,0.7475,EasyEnsembleClassifier imbalance-aware ensemble
2,Block 2: Makkena & Natarajan,AdaBoost,baseline,None,Yes,No,0.6426,0.5340,0.6404,0.5859,0.7344,Baseline AdaBoostClassifier + SMOTE
3,Block 3: Ma & Li,Balanced Bagging,baseline,None,No,No,0.6645,0.5571,0.6579,0.5806,0.7351,BalancedBaggingClassifier imbalance-aware ense...
4,Block 2: Makkena & Natarajan,Logistic Regression,tuned,StandardScaler,Yes,Yes,0.6118,0.5645,0.6140,0.5769,0.7856,GridSearchCV over LogisticRegression C and pen...
5,Block 3: Ma & Li,RUSBoost,baseline,None,No,No,0.6294,0.4893,0.6842,0.5714,0.7306,RUSBoostClassifier imbalance-aware boosting
6,Block 1: Hanif & Khan,SVM (RBF),baseline,StandardScaler,Yes,No,0.6184,0.5321,0.6404,0.5591,0.7497,SVC with RBF kernel + scaling + SMOTE recreation
7,Block 2: Makkena & Natarajan,Random Forest,tuned,None,Yes,Yes,0.6756,0.5685,0.6491,0.5556,0.7314,RandomizedSearchCV over RandomForestClassifier...
8,Block 2: Makkena & Natarajan,Naive Bayes,baseline,None,Yes,No,0.5658,0.5495,0.5614,0.5536,0.7175,Baseline GaussianNB + SMOTE
9,Block 2: Makkena & Natarajan,XGBoost,tuned,None,Yes,Yes,0.6842,0.5305,0.6667,0.5476,0.7262,RandomizedSearchCV over XGBClassifier settings...


,Section,Model,Variant,CV F1,Holdout F1,ROC-AUC,Notes
0,Block 2: Makkena & Natarajan,Logistic Regression,baseline,0.5458,0.5918,0.7826,Baseline LogisticRegression + scaling + SMOTE


,Section,Model,Variant,CV F1,Holdout F1,ROC-AUC,Notes
0,Block 2: Makkena & Natarajan,Logistic Regression,baseline,0.5458,0.5918,0.7826,Baseline LogisticRegression + scaling + SMOTE
1,Block 3: Ma & Li,Easy Ensemble,baseline,0.5604,0.5859,0.7475,EasyEnsembleClassifier imbalance-aware ensemble
2,Block 1: Hanif & Khan,SVM (RBF),baseline,0.5321,0.5591,0.7497,SVC with RBF kernel + scaling + SMOTE recreation


{'results_csv': '/Users/samyabrataroy/Downloads/Spring_Internship_2026/output/literature_recreation_cleaned_ilpd.csv', 'best_model_pickle': '/Users/samyabrataroy/Downloads/Spring_Internship_2026/output/best_literature_recreation_cleaned_ilpd.pkl', 'best_model': 'Logistic Regression', 'best_model_variant': 'baseline'}


“This notebook recreates literature-inspired pipelines on the cleaned ILPD dataset under a unified 5-fold evaluation framework. Subsequent experiments involving GMM-induced noise and proposed uncertainty-aware models will be conducted in a separate notebook.”
